# Pipeline Unificado — Sistema Híbrido de Omni-Ensemble (SEE)

Este notebook é um **executor fino**: toda a lógica vive em `src/` (fonte única de verdade). Aqui só importamos e orquestramos T1→T8 sobre as bases reais.

- **T1** carregamento real + pré-processamento (`dataset_loader`)
- **T2** pool de modelos individuais, sem ensembles (`train_pool`)
- **T3** SES-GA mono-objetivo (`ses_ga_single`)
- **T4** SES-GA multi-objetivo / NSGA-II (`ses_ga_multi`)
- **T5** seleção dinâmica DES (`des_dynamic`)
- **T6** métricas (`evaluator`) · **T7** Friedman + Bonferroni-Dunn (`statistical_tests`)
- **Combinação** (`combination`): média simples (baseline); ponderada por competência entra na Etapa 1.

In [ ]:
# Bootstrap: garante a raiz do projeto no path e como CWD
import os, sys
root = os.getcwd()
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'src')):
    root = os.path.dirname(root)
assert os.path.isdir(os.path.join(root, 'src')), 'pasta src/ nao encontrada'
os.chdir(root); sys.path.insert(0, root)
print('Raiz do projeto:', root)

In [ ]:
import numpy as np
import pandas as pd

from src.dataset_loader import run_pipeline, datasets_with_pool, load_pool_arrays, load_xy
from src.train_pool import run_train_pool, build_pool
from src.ses_ga_single import run_ses_ga_pipeline
from src.ses_ga_multi import run_ses_ga_multi_pipeline
from src.des_dynamic import DynamicEnsembleSelector
from src.combination import combine_mean
from src.evaluator import evaluate_all, smape, mre, mase, nse, cod
from src.statistical_tests import friedman_test, bonferroni_dunn_test, plot_cd_diagram
print('Pool base:', list(build_pool().keys()))

## T1 — Dados reais e pré-processamento

In [ ]:
splits = run_pipeline()

## T2 — Pool de modelos base (sem ensembles)

In [ ]:
pool = run_train_pool()

## T3 — SES-GA mono-objetivo

In [ ]:
t3 = run_ses_ga_pipeline(verbose=False)
print({d: t3[d]['n_models_selected'] for d in t3}, '(# modelos selecionados por base)')

## T4 — SES-GA multi-objetivo (R² × parcimônia)

In [ ]:
t4 = run_ses_ga_multi_pipeline(verbose=False)
print({d: t4[d]['best_balanced']['n_models'] for d in t4}, '(# modelos na solucao balanceada)')

## T5–T8 — Integração: DES, combinação, métricas e testes de hipótese

In [ ]:
base_names = pool[next(iter(pool))]['model_names']
method_names = base_names + ['Static', 'SES-GA', 'SES-GA-Multi', 'DES']

summary_rows, matrix_rows = [], []
for d in datasets_with_pool():
    pred_train, pred_test, y_train, y_test = load_pool_arrays(d)
    X_train, X_test, _, _ = load_xy(d)

    des = DynamicEnsembleSelector(k=7, threshold=0.3)
    des.fit(X_train, pred_train, y_train)
    yp = {
        'DES':          des.predict(X_test, pred_test),
        'Static':       combine_mean(pred_test, None),
        'SES-GA':       combine_mean(pred_test, t3[d]['best_chromosome']),
        'SES-GA-Multi': combine_mean(pred_test, t4[d]['best_balanced']['chromosome']),
    }
    for name, pred in yp.items():
        summary_rows.append({'dataset': d, 'method': name,
                             'sMAPE': smape(y_test, pred), 'MRE': mre(y_test, pred),
                             'MASE': mase(y_test, pred), 'NSE': nse(y_test, pred),
                             'COD': cod(y_test, pred)})
    base_err = [smape(y_test, pred_test[:, i]) for i in range(pred_test.shape[1])]
    matrix_rows.append(base_err + [smape(y_test, yp['Static']), smape(y_test, yp['SES-GA']),
                                   smape(y_test, yp['SES-GA-Multi']), smape(y_test, yp['DES'])])

summary_df = pd.DataFrame(summary_rows)
display(summary_df.pivot(index='dataset', columns='method', values='sMAPE').round(3))

results_matrix = np.array(matrix_rows, dtype=float)
print('Matriz de erro sMAPE (bases x metodos):', results_matrix.shape)
display(pd.DataFrame(results_matrix, columns=method_names, index=datasets_with_pool()).round(2))

In [ ]:
# Controle = metodo proposto. Hoje SES-GA-Multi; na Etapa 1 passa a ser
# 'SES-GA-Multi + combinacao ponderada por competencia'.
control = 'SES-GA-Multi'
fr = friedman_test(results_matrix, method_names)
if fr['significant']:
    bonferroni_dunn_test(fr, control_model=control, n_datasets=results_matrix.shape[0])
else:
    print('[INFO] Friedman nao significativo — post-hoc nao aplicado.')
plot_cd_diagram(fr, control_model=control, n_datasets=results_matrix.shape[0])